本实验用到的葡萄酒数据集中Sklearn库自带的，该数据集包含178条数据，每条数据拥有14个字段，其中13个是特征值，1个是表示分类的目标值。
该数据集的特征值和目标值的含义如图所示，其中Atrribute Information部分展示的是13个特征值的具体含义，如Alcohol字段表示该类葡萄酒的酒精含量，而class则是目标值，该数据集里的数据被划分为3类。
下面将演示用神经网络模型分类预测葡萄酒数据的方法。具体是先把该数据集划分成训练集和测试集，再用训练集训练神经网络模型，并用训练好的模型预测测试集的葡萄酒分类。

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 加载WINE数据集
data = load_wine()
print('wine_datas.DESCR:', data.DESCR)
sample = pd.concat([pd.DataFrame(data.data), pd.DataFrame(data.target)], axis=1)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
# 显示表格的前5行
print(sample.head)

# 数据预处理
feature = data.data     # 获取数据特征，共178个样本，每个样本为13维向量
target = data.target    # 获取数据标签，共3类，分别为0，1，2
scaler = StandardScaler()
feature = scaler.fit_transform(feature)
# 划分训练集和测试集
feature_train, feature_test, y_train, y_test = train_test_split(feature, target, test_size=0.3, random_state=42)

# 转换为PyTorch张量
feature_train = torch.from_numpy(feature_train).float()
target_train =  torch.from_numpy(y_train).long()
feature_test = torch.from_numpy(feature_test).float()
target_test =  torch.from_numpy(y_test).long()

# 定义神经网络
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(13, 64)    #模型接受的入参维度为13，与葡萄酒数据集的特征值个数相符
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 3)     #模型的出参维度为3，与葡萄酒数据集的分类数量相符
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x)) 
        x = self.fc3(x)
        return x
net =  Net()

# 定义损失函数与优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# 训练神经网络
for epoch in range(200):
    optimizer.zero_grad()
    output = net(feature_train)
    loss = criterion(output, target_train)
    loss.backward()
    optimizer.step()
    if(epoch+1)% 20 == 0:
        print(f'Epoch [{epoch + 1} / 200], Train Loss: {loss.item():.4f}')

# 在测试集上进行预测
with torch.no_grad():
    output = net(feature_test)
    _, predicted = torch.max(output, 1)
    total = target_test.size(0)
    correct = (predicted == target_test).sum().item()
    accuracy = correct / total
    print('Accuracy on test set: %.2f%%' % (accuracy * 100))

在模型训练过程中，能看到经多轮训练后，输出的损失值数据越来越小，模型的预测精度会不断提升
Epoch [20/200], Train Loss: 0.8167
.......
Epoch [200/200], Train Loss: 0.0067

完成训练后，模型用于预测测试集，输出结果显示，模型能准确地预测测试集数据。
Accuracy on test set: 100.00%